# M2: 제약형 CLV-경제속성 임베딩 (Dunnhumby, seed 42)

사용자는 `q_C × Unit(W[q_N,q_V])`, 상품은 `Unit(W[전체 가격 위치, 카테고리 내 가격 위치])`로 표현합니다. 자유 상품 반응 임베딩과 tanh는 사용하지 않습니다. ID 64차원과 CLV 4차원을 layer-0에서 결합하여 하나의 이진 LightGCN·BPR·optimizer로 공동학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/clv-m2-lightgcn-runner
!git clone --branch feat/m2-joint-nv-lightgcn https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout e940af1
!git rev-parse HEAD

In [ ]:
import json
import torch
from lightgcn_clv_constrained_economic_embedding import (
    configure_constrained_economic_run,
    preflight_summary,
    run_constrained_economic_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_constrained_economic_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_constrained_economic_screen(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표')
display(result_df)
print('2) 대조군별 전체 성과 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) CLV 구간별 Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 실제 점수 영향력')
display(pd.DataFrame(result_df.attrs['score_diagnostics']))
print('5) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('6) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))